In [1]:
from pathlib import Path

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

QWEN_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
QWEN_LANGUAGE = "English"
QWEN_SPEAKER = "Ryan"
QWEN_INSTRUCT = None  # e.g., "Speak in a cheerful and positive tone."
OUTPUT_PATH = "outputs/qwen_tts.wav"


/Users/aps/UPenn/vivaprox/notebooks/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/bin/sh: sox: command not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


## Qwen (TTS)

Reference: https://pypi.org/project/qwen-tts/

Notes:
- Use the repo's Python 3.10 environment (required by `accelerate`).
- First run downloads model weights.


In [2]:
use_cuda = torch.cuda.is_available()
if not use_cuda:
    print("Warning: CUDA not available; CPU inference may be very slow.")

device_map = "cuda:0" if use_cuda else "cpu"
dtype = torch.bfloat16 if use_cuda else torch.float32

kwargs = {"device_map": device_map, "dtype": dtype}
if use_cuda:
    kwargs["attn_implementation"] = "flash_attention_2"

model = Qwen3TTSModel.from_pretrained(QWEN_MODEL_ID, **kwargs)


Fetching 4 files: 100%|████| 4/4 [00:11<00:00,  2.93s/it]


In [3]:
model.get_supported_speakers()


['aiden',
 'dylan',
 'eric',
 'ono_anna',
 'ryan',
 'serena',
 'sohee',
 'uncle_fu',
 'vivian']

In [4]:
wavs, sr = model.generate_custom_voice(
    text="Hello from Qwen. This is a quick TTS smoke test.",
    language=QWEN_LANGUAGE,
    speaker=QWEN_SPEAKER,
    instruct=QWEN_INSTRUCT or "",
)

Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
sf.write(OUTPUT_PATH, wavs[0], sr)

display(Audio(wavs[0], rate=sr))


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
